In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

from concurrent.futures import ThreadPoolExecutor
import optuna
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

import shap

In [ ]:


input_path = Path("derived") / "dataset.xlsx"  # Path to the input workbook.
output_dir = Path("encoding_sensitivity")  # Folder for generated results.
output_dir.mkdir(exist_ok=True)
seed = 0  # Random seed for reproducibility.

features = [  # Features.
    "wt% Cu", "wt% Support1", "χ Support1", "wt% Support2", "χ Support2",
    "wt% Support3", "χ Support3", "wt% Promoter", "χ Promoter",
    "Synthesis Method", "Calc T", "Calc D", "H2/CO2", "GHSV", "P", "T",
    "SBET", "SACu", "dCu",
]
point_id = "Data point number"  # Unique experimental-record identifier column.
calc_columns = ["Calc T", "Calc D"]  # Calcination temperature and duration columns.
flag_columns = [f"{column} imputed" for column in calc_columns]  # Columns indicating missing calcination values. Current column name.
data = pd.read_excel(input_path, sheet_name="data", engine="openpyxl")  # Input records for this analysis.
if data[point_id].isna().any() or not data[point_id].is_unique:
    raise ValueError("Data point numbers must be present and unique.")
X = data[features].apply(pd.to_numeric).astype(float)  # Predictor matrix.
if not np.isfinite(X.to_numpy()).all():
    raise ValueError("Encoded model inputs must be finite.")
for column, flag in zip(calc_columns, flag_columns):  # Current column name; Indicator for the current record or column.
    if not data[flag].isin([False, True]).all():
        raise ValueError(f"{flag} must contain Boolean values.")
    X[column] = X[column].mask(data[flag].astype(bool))

train_idx, test_idx = train_test_split(data.index, test_size=0.2, random_state=seed)  # Indices of the development records; Indices of the held-out records.
train_idx, test_idx = pd.Index(train_idx), pd.Index(test_idx)
split = data[[point_id, "Catalyst number", "Reference DOI"]].assign(Split="Test")  # Training and held-out partition assignments.
split.loc[train_idx, "Split"] = "Train"
print(f"Records: {len(data)} | Train: {len(train_idx)} | Test: {len(test_idx)}")


def calcination_fill(X_fit):  # Return rounded calcination means for the training rows.
    columns = X_fit.columns.intersection(calc_columns)  # Columns included in the operation.
    means = X_fit[columns].mean()  # Calculated means for imputation or plotting.
    if means.isna().any():
        raise ValueError("A training fold lacks observed calcination values.")
    return np.floor(means + 0.5)




In [ ]:


optuna.logging.set_verbosity(optuna.logging.WARNING)
n_splits = 5  # Number of cross-validation folds.
cv_workers = 5  # Cross-validation workers.
folds = list(KFold(n_splits=n_splits, shuffle=True, random_state=seed).split(train_idx))  # Cross-validation splits.
model_defaults = {"objective": "reg:squarederror", "tree_method": "hist", "random_state": seed, "n_jobs": 1}  # Fixed estimator options.


def prediction(model, values, upper=None):  # Predict responses and enforce the configured prediction bounds.
    return np.clip(model.predict(values), 0, upper)


def regression_scores(y, pred):  # Compute regression performance from paired observed and predicted values.
    return {"R2": r2_score(y, pred), "RMSE": np.sqrt(mean_squared_error(y, pred)), "MAE": mean_absolute_error(y, pred)}


def encode_methods(train, evaluation, representation):  # Build the requested integer or one-hot synthesis representation.
    if representation == "Ordinal":
        return train, evaluation

    name = "Synthesis Method"  # Current item name.
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)  
    train_methods = encoder.fit_transform(train[[name]])  
    evaluation_methods = encoder.transform(evaluation[[name]])  
    columns = encoder.get_feature_names_out()  # Columns included in the operation.

    def combine(frame, values):  # Join the fold-level results for one synthesis encoding.
        indicators = pd.DataFrame(values, index=frame.index, columns=columns)  
        return pd.concat([frame.drop(columns=name), indicators], axis=1)

    return combine(train, train_methods), combine(evaluation, evaluation_methods)


def fit_regression(target, columns, trials, representation, upper=None, early_stop=False):  # Tune and fit an XGBoost regressor using the training folds.
    y = data[target].astype(float)  
    if not np.isfinite(y.to_numpy()).all():
        raise ValueError(f"{target} must contain finite observations.")
    X_fit = X.loc[train_idx, columns]  # Predictors used to fit the estimator.
    cv_sets = []  # Prepared cross-validation partitions.
    for fold, (i, j) in enumerate(folds, start=1):  # Cross-validation fold; Loop position; Second loop position.
        X_tr, X_va = X_fit.iloc[i], X_fit.iloc[j]  # Training-fold predictors; Validation-fold predictors.
        fill = calcination_fill(X_tr)  # Rounded imputation values from the fitting data.
        X_tr, X_va = encode_methods(X_tr.fillna(fill), X_va.fillna(fill), representation)
        cv_sets.append((fold, X_tr, y.loc[X_tr.index], X_va, y.loc[X_va.index]))

    def score_fold(params, fold_data):  # Fit and evaluate the regressor for one validation split.
        fold, X_tr, y_tr, X_va, y_va = fold_data  # Cross-validation fold; Training-fold predictors; Training-fold responses; Validation-fold predictors; Validation-fold responses.
        if early_stop:
            model = XGBRegressor(**model_defaults, **params, n_estimators=1000, early_stopping_rounds=20)  # Estimator for the current task.
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            rounds = model.best_iteration + 1  # Number of boosting iterations.
        else:
            model = XGBRegressor(**model_defaults, **params)
            model.fit(X_tr, y_tr)
            rounds = params["n_estimators"]
        return {
            "Fold": fold, "Rounds": rounds, **regression_scores(y_va, prediction(model, X_va, upper)),
            "Train RMSE": regression_scores(y_tr, prediction(model, X_tr, upper))["RMSE"],
        }

    with ThreadPoolExecutor(max_workers=cv_workers) as pool:  
        def objective(trial):  # Return cross-validation error for the current Optuna trial.
            if early_stop:
                params = {  # Estimator hyperparameters.
                    "max_depth": trial.suggest_int("max_depth", 2, 5),
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                    "min_child_weight": trial.suggest_int("min_child_weight", 10, 30),
                    "subsample": trial.suggest_float("subsample", 0.5, 0.85),
                    "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 0.85),
                    "reg_alpha": trial.suggest_float("reg_alpha", 1e-2, 10, log=True),
                    "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 50, log=True),
                    "gamma": trial.suggest_float("gamma", 0, 5),
                }
            else:
                params = {
                    "n_estimators": trial.suggest_int("n_estimators", 50, 300),
                    "max_depth": trial.suggest_int("max_depth", 3, 10),
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                    "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                    "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                }
            scores = list(pool.map(lambda s: score_fold(params, s), cv_sets))  # Scores for the current evaluation.
            trial.set_user_attr("fold_scores", scores)
            return float(np.mean([row["RMSE"] for row in scores]))  

        study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed))  # Optuna hyperparameter search.
        print(f"Searching {target} ({representation}): {trials} trials × {n_splits} folds", flush=True)
        study.optimize(objective, n_trials=trials)

    params = study.best_params.copy()  # Estimator hyperparameters.
    fold_scores = pd.DataFrame(study.best_trial.user_attrs["fold_scores"]).assign(Target=target)  
    if early_stop:
        params["n_estimators"] = max(1, int(1.1 * fold_scores["Rounds"].mean()))
    fill = calcination_fill(X_fit)
    fitted_X = X_fit.fillna(fill)  
    test_X = X.loc[test_idx, columns].fillna(fill)  
    fitted_X, test_X = encode_methods(fitted_X, test_X, representation)
    model = XGBRegressor(**model_defaults, **params)  # Estimator for the current task.
    model.fit(fitted_X, y.loc[train_idx])

    predicted = pd.Series(index=data.index, dtype=float, name="Predicted")  # Predicted response values.
    metrics = []  # Performance metrics.
    for subset, values in [("Train", fitted_X), ("Test", test_X)]:  # Subset; Values used in the current calculation.
        pred = prediction(model, values, upper) 
        predicted.loc[values.index] = pred
        metrics.append({"Target": target, "Split": subset, "N": len(pred), **regression_scores(y.loc[values.index], pred)})
    print(f"{target} | trial {study.best_trial.number} | CV RMSE {study.best_value:.4f}", flush=True)
    return model, fitted_X, test_X, fill, params, predicted, pd.DataFrame(metrics), fold_scores, study




In [ ]:

summaries, fold_results, all_predictions, all_importances, parameters = [], [], [], [], []  
trial_count = 100  
unit_metrics = ["R2", "RMSE", "MAE"]  #
for representation in ("Ordinal", "One-hot"): 
    model, fit_X, test_X, _, params, predicted, metrics, fold_scores, _ = fit_regression(  # Estimator for the current task; Fit x; Test x; Estimator hyperparameters; Predicted response values; Performance metrics; Fold scores.
        "STY", features, trial_count, representation, early_stop=True,
    )
    summaries.append(metrics.assign(Encoding=representation))
    fold_results.append(fold_scores.assign(Encoding=representation))
    all_predictions.append(split.assign(Actual=data["STY"], Predicted=predicted, Encoding=representation))
    parameters.append({"Encoding": representation, **params})
    background = shap.sample(fit_X, min(100, len(fit_X)), random_state=seed)  
    explanation = shap.TreeExplainer(model, data=background, feature_perturbation="interventional")  # TreeSHAP values and their feature data.
    shap_values = pd.DataFrame(explanation(test_X, check_additivity=False).values, columns=test_X.columns)  
    method_columns = [column for column in shap_values if column.startswith("Synthesis Method_")]  
    if method_columns:
        shap_values["Synthesis Method"] = shap_values[method_columns].sum(axis=1)
        shap_values = shap_values.drop(columns=method_columns)
    importance = shap_values.abs().mean().sort_values(ascending=False).rename("Mean absolute SHAP") 
    all_importances.append(importance.rename_axis("Feature").reset_index().assign(
        Encoding=representation, Rank=range(1, len(importance) + 1),
    ))

summary = pd.concat(summaries, ignore_index=True)  # Summary.
fold_scores = pd.concat(fold_results, ignore_index=True)
cv_stats = fold_scores.groupby("Encoding", sort=False)[unit_metrics].agg(["mean", "std"])  # Cross-validation stats.
cv_stats.columns = [f"{metric} {aggregation}" for metric, aggregation in cv_stats.columns]  # Metric; Aggregation.
ordinal = fold_scores.loc[fold_scores["Encoding"].eq("Ordinal")].set_index("Fold")  
nominal = fold_scores.loc[fold_scores["Encoding"].eq("One-hot")].set_index("Fold") 
paired_deltas = nominal[unit_metrics] - ordinal[unit_metrics]  
destination = output_dir / "synthesis_method_encoding.xlsx"  
with pd.ExcelWriter(destination, engine="openpyxl") as writer:  # Excel writer for the result workbook.
    summary.to_excel(writer, sheet_name="holdout_metrics", index=False)
    cv_stats.to_excel(writer, sheet_name="cv_summary")
    fold_scores.to_excel(writer, sheet_name="cv_folds", index=False)
    paired_deltas.to_excel(writer, sheet_name="onehot_minus_ordinal")
    pd.concat(all_predictions).to_excel(writer, sheet_name="predictions", index=False)
    pd.concat(all_importances).to_excel(writer, sheet_name="holdout_shap", index=False)
    pd.DataFrame(parameters).to_excel(writer, sheet_name="hyperparameters", index=False)
display(cv_stats.round(4))
display(summary.round(4))
